# 환율 데이터 수집 파이프라인 (Korea Exim Exchange Rate)

한국수출입은행(Korea Exim) 환율 API를 호출하여 일별 환율 데이터를 수집하고 Delta 테이블에 저장하는 재사용 가능한 파이프라인입니다. 
Job Parameter(widgets)를 통해 **조회 일자, API 키, 대상 테이블 경로, 데이터 구분**을 외부에서 주입할 수 있습니다.

In [0]:
%skip
# ============================================================
# 🔒 Secrets 설정 (최초 1회만 실행 → 이후 주석 처리 또는 셀 삭제)
# ============================================================
# Databricks SDK를 사용해 Secret Scope 생성, API Key 저장, SP 권한 부여를 수행합니다.
# Notebook 인증 컨텍스트를 사용하므로 별도 토큰 불필요.

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists
from databricks.sdk.service.workspace import AclPermission

w = WorkspaceClient()

# --- 설정값 (필요에 따라 수정) ---
SECRET_SCOPE = "exchange-rate-dev"          # 생성할 Secret Scope 이름
SECRET_KEY   = "korea-exim-api-key"                  # Secret Key 이름
API_KEY_VALUE = "38fIqptdTHKN7NJVFg5spRgXTm47sFz2"  # 한국수출입은행 API 인증키
SP_CLIENT_ID = "d1185284-4f87-4c40-9e68-ddb31350c87e"  # 권한 부여할 SP Client ID 로 교체
# --------------------------------

# 1. Secret Scope 생성 (이미 존재하면 skip)
try:
    w.secrets.create_scope(scope=SECRET_SCOPE)
    print(f"✅ Secret Scope '{SECRET_SCOPE}' 생성 완료")
except ResourceAlreadyExists:
    print(f"ℹ️ Secret Scope '{SECRET_SCOPE}' 이미 존재함 (skip)")
except Exception as e:
    print(f"❌ Secret Scope 생성 실패: {e}")

# 2. API Key를 Secret에 저장
w.secrets.put_secret(
    scope=SECRET_SCOPE,
    key=SECRET_KEY,
    string_value=API_KEY_VALUE
)
print(f"✅ Secret '{SECRET_KEY}' 저장 완료 (scope={SECRET_SCOPE})")

# 3. Service Principal에 READ 권한 부여
w.secrets.put_acl(
    scope=SECRET_SCOPE,
    principal=SP_CLIENT_ID,
    permission=AclPermission.READ
)
print(f"✅ SP '{SP_CLIENT_ID}' → READ 권한 부여 완료")

# 4. 현재 ACL 확인
print(f"\n📋 현재 ACL 목록 ({SECRET_SCOPE}):")
for acl in w.secrets.list_acls(scope=SECRET_SCOPE):
    print(f"   - principal: {acl.principal}, permission: {acl.permission}")

ℹ️ Secret Scope 'exchange-rate-dev' 이미 존재함 (skip)
✅ Secret 'korea-exim-api-key' 저장 완료 (scope=exchange-rate-dev)
✅ SP 'd1185284-4f87-4c40-9e68-ddb31350c87e' → READ 권한 부여 완료

📋 현재 ACL 목록 (exchange-rate-dev):
   - principal: d1185284-4f87-4c40-9e68-ddb31350c87e, permission: AclPermission.READ
   - principal: hyemi.jeong@data-dynamics.io, permission: AclPermission.MANAGE


In [0]:
# ============================================================
# Job Parameters (Widgets) - 외부에서 주입 가능한 파라미터 정의
# ============================================================
# 이 셀의 widget 값들은 Job 실행 시 "매개변수" 탭에서 덮어쓸 수 있습니다.
# Databricks Job에서 key=value 형태로 전달하면 자동으로 widget에 반영됩니다.
#workspace.default
# 1. 조회 일자 (YYYYMMDD 형식, 빈 값이면 오늘 날짜)
dbutils.widgets.text("search_date", "", "조회 일자 (YYYY-MM-DD, 공백=오늘)")

# 2. API 인증키 (Databricks Secrets에서 로드)
#    - secret_scope: Databricks Secret Scope 이름
#    - secret_key:   Secret Scope 내 key 이름
#    - 실제 API Key는 Secrets에 저장하고 dbutils.secrets.get()로 불러옵니다.
dbutils.widgets.text("secret_scope", "exchange-rate-dev", "Secret Scope 이름")
dbutils.widgets.text("secret_key", "korea-exim-api-key", "Secret Key 이름")

# 3. 데이터 구분 (AP01: 일별환율, AP02: 환율추이)
dbutils.widgets.text("data_type", "AP01", "데이터 구분 (AP01/AP02)")

# 4. 대상 카탈로그
dbutils.widgets.text("target_catalog", "workspace", "대상 카탈로그")

# 5. 대상 스키마
dbutils.widgets.text("target_schema", "default", "대상 스키마")

# 6. 대상 테이블명
dbutils.widgets.text("target_table", "daily_exchange_rate", "대상 테이블명")

# 7. 모드 (append: 추가, overwrite: 전체 덮어쓰기)
dbutils.widgets.text("write_mode", "append", "쓰기 모드 (append/overwrite)")

# Secrets에서 API Key 로드
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

print(f"✅ Job Parameters 정의 완료")
print(f"🔒 API Key 로드 완료 (scope={secret_scope}, key={secret_key})")

✅ Job Parameters 정의 완료
🔒 API Key 로드 완료 (scope=exchange-rate-dev, key=korea-exim-api-key)


In [0]:
# ============================================================
# 파라미터 읽기 및 API 호출 준비
# ============================================================
from datetime import datetime, timedelta
import requests
import json

# Widget 값 읽기 (api_key는 Cell 2에서 Secrets로 로드됨)
search_date = dbutils.widgets.get("search_date")

run_date = dbutils.widgets.get("search_date")
search_date = datetime.strptime(run_date, "%Y-%m-%d").strftime("%Y%m%d")

data_type = dbutils.widgets.get("data_type")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")
write_mode = dbutils.widgets.get("write_mode")

# 조회 일자 처리 (빈 값이면 오늘 날짜 사용)
if not search_date:
    search_date = datetime.now().strftime("%Y%m%d")

# API 엔드포인트 (한국수출입은행 OpenAPI)
API_URL = "https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON"

# 요청 파라미터 구성
params = {
    "authkey": api_key,
    "searchdate": search_date,
    "data": data_type
}

print(f"📅 조회 일자: {search_date}")
print(f"📊 데이터 구분: {data_type} ({'일별환율' if data_type == 'AP01' else '환율추이'})")
print(f"🗂️ 대상 테이블: {target_catalog}.{target_schema}.{target_table}")
print(f"📝 쓰기 모드: {write_mode}")
print(f"🔗 API URL: {API_URL}")

📅 조회 일자: 20261006
📊 데이터 구분: AP01 (일별환율)
🗂️ 대상 테이블: workspace.default.daily_exchange_rate
📝 쓰기 모드: append
🔗 API URL: https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON


In [0]:
# ============================================================
# API 호출 및 응답 처리
# ============================================================

response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()

raw_data = response.json()

print(f"HTTP Status: {response.status_code}")
print(f"응답 건수: {len(raw_data)}건")

if len(raw_data) == 0:
    print("⚠️ 조회 결과가 없습니다. (휴일이거나 API 미갱신)")
else:
    print(f"✅ 데이터 수신 완료: {len(raw_data)}건")
    print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

HTTP Status: 200
응답 건수: 23건
✅ 데이터 수신 완료: 23건
{
  "result": 1,
  "cur_unit": "AED",
  "ttb": "362.12",
  "tts": "369.43",
  "deal_bas_r": "365.78",
  "bkpr": "365",
  "yy_efee_r": "0",
  "ten_dd_efee_r": "0",
  "kftc_bkpr": "365",
  "kftc_deal_bas_r": "365.78",
  "cur_nm": "아랍에미리트 디르함"
}


In [0]:
# ============================================================
# JSON 응답 → Spark DataFrame 변환
# ============================================================
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)
from pyspark.sql import functions as F

if len(raw_data) > 0:
    # 스키마 정의 (한국수출입은행 환율 API 응답 필드)
    schema = StructType([
        StructField("result", IntegerType(), True),           # 결과코드 (1=성공)
        StructField("cur_unit", StringType(), True),            # 통화코드 (USD, EUR, ...)
        StructField("ttb", StringType(), True),                # 전신환매도율
        StructField("tts", StringType(), True),                # 전신환매입율
        StructField("deal_bas_r", StringType(), True),          # 매매기준율
        StructField("bkpr", StringType(), True),               # 장부가격
        StructField("yy_efee_r", StringType(), True),          # 년환가격
        StructField("ten_dd_efee_r", StringType(), True),      # 10일환가격
        StructField("kftc_bkpr", StringType(), True),          # 서울외환중개 장부가격
        StructField("kftc_deal_bas_r", StringType(), True),   # 서울외환중개 매매기준율
        StructField("cur_nm", StringType(), True),            # 통화명 (미국 달러, ...)
    ])
    
    # Spark DataFrame 생성
    df_raw = spark.createDataFrame(raw_data, schema=schema)
    
    # 문자형 숫자 → Double 변환 및 메타데이터 추가
    df = (df_raw
        .withColumn("deal_bas_r_num", F.regexp_replace("deal_bas_r", ",", "").cast(DoubleType()))
        .withColumn("ttb_num", F.regexp_replace("ttb", ",", "").cast(DoubleType()))
        .withColumn("tts_num", F.regexp_replace("tts", ",", "").cast(DoubleType()))
        .withColumn("search_date", F.lit(search_date))
        .withColumn("reg_dt", F.current_timestamp())
    )
    
    print(f"DataFrame 생성 완료: {df.count()}행")
    display(df)
else:
    print("DataFrame을 생성할 데이터가 없습니다.")

DataFrame 생성 완료: 23행


result,cur_unit,ttb,tts,deal_bas_r,bkpr,yy_efee_r,ten_dd_efee_r,kftc_bkpr,kftc_deal_bas_r,cur_nm,deal_bas_r_num,ttb_num,tts_num,search_date,ingestion_ts
1,AED,362.12,369.43,365.78,365,0,0,365,365.78,아랍에미리트 디르함,365.78,362.12,369.43,20261007,2026-10-07T03:31:33.354Z
1,AUD,928.31,947.06,937.69,937,0,0,937,937.69,호주 달러,937.69,928.31,947.06,20261007,2026-10-07T03:31:33.354Z
1,BHD,"3,527.47","3,598.74","3,563.11","3,563",0,0,"3,563","3,563.11",바레인 디나르,3563.11,3527.47,3598.74,20261007,2026-10-07T03:31:33.354Z
1,BND,"1,041.02","1,062.05","1,051.54","1,051",0,0,"1,051","1,051.54",브루나이 달러,1051.54,1041.02,1062.05,20261007,2026-10-07T03:31:33.354Z
1,CAD,935.83,954.74,945.29,945,0,0,945,945.29,캐나다 달러,945.29,935.83,954.74,20261007,2026-10-07T03:31:33.354Z
1,CHF,"1,598.22","1,630.51","1,614.37","1,614",0,0,"1,614","1,614.37",스위스 프랑,1614.37,1598.22,1630.51,20261007,2026-10-07T03:31:33.354Z
1,CNH,198.3,202.31,200.31,200,0,0,200,200.31,위안화,200.31,198.3,202.31,20261007,2026-10-07T03:31:33.354Z
1,DKK,200.3,204.35,202.33,202,0,0,202,202.33,덴마아크 크로네,202.33,200.3,204.35,20261007,2026-10-07T03:31:33.354Z
1,EUR,"1,497.14","1,527.39","1,512.27","1,512",0,0,"1,512","1,512.27",유로,1512.27,1497.14,1527.39,20261007,2026-10-07T03:31:33.354Z
1,GBP,"1,765","1,800.65","1,782.83","1,782",0,0,"1,782","1,782.83",영국 파운드,1782.83,1765.0,1800.65,20261007,2026-10-07T03:31:33.354Z


In [0]:
# ============================================================
# Delta 테이블에 저장 (파라미터 기반 동적 경로)
# ============================================================
from pyspark.sql import functions as F

full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

if len(raw_data) > 0:
    # 카탈로그/스키마 생성 (존재하지 않을 경우)
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")
    
    # Delta 테이블에 쓰기 (write_mode 파라미터 사용)
    print(f"Writing to {full_table_name} (mode={write_mode})...")
    
    (df.write
        .mode(write_mode)
        .format("delta")
        .saveAsTable(full_table_name))
    
    print(f"✅ 저장 완료: {full_table_name}")
    
    # 저장 결과 확인
    row_count = spark.table(full_table_name).count()
    print(f"📊 테이블 전체 건수: {row_count}건")
    display(spark.table(full_table_name).limit(20))
else:
    print("⚠️ 저장할 데이터가 없습니다. (휴일 또는 API 응답 없음)")

Writing to workspace.default.daily_exchange_rate (mode=append)...
✅ 저장 완료: workspace.default.daily_exchange_rate
📊 테이블 전체 건수: 23건


result,cur_unit,ttb,tts,deal_bas_r,bkpr,yy_efee_r,ten_dd_efee_r,kftc_bkpr,kftc_deal_bas_r,cur_nm,deal_bas_r_num,ttb_num,tts_num,search_date,ingestion_ts
1,AED,362.12,369.43,365.78,365,0,0,365,365.78,아랍에미리트 디르함,365.78,362.12,369.43,20261007,2026-10-07T03:31:56.002Z
1,AUD,928.31,947.06,937.69,937,0,0,937,937.69,호주 달러,937.69,928.31,947.06,20261007,2026-10-07T03:31:56.002Z
1,BHD,"3,527.47","3,598.74","3,563.11","3,563",0,0,"3,563","3,563.11",바레인 디나르,3563.11,3527.47,3598.74,20261007,2026-10-07T03:31:56.002Z
1,BND,"1,041.02","1,062.05","1,051.54","1,051",0,0,"1,051","1,051.54",브루나이 달러,1051.54,1041.02,1062.05,20261007,2026-10-07T03:31:56.002Z
1,CAD,935.83,954.74,945.29,945,0,0,945,945.29,캐나다 달러,945.29,935.83,954.74,20261007,2026-10-07T03:31:56.002Z
1,CHF,"1,598.22","1,630.51","1,614.37","1,614",0,0,"1,614","1,614.37",스위스 프랑,1614.37,1598.22,1630.51,20261007,2026-10-07T03:31:56.002Z
1,CNH,198.3,202.31,200.31,200,0,0,200,200.31,위안화,200.31,198.3,202.31,20261007,2026-10-07T03:31:56.002Z
1,DKK,200.3,204.35,202.33,202,0,0,202,202.33,덴마아크 크로네,202.33,200.3,204.35,20261007,2026-10-07T03:31:56.002Z
1,EUR,"1,497.14","1,527.39","1,512.27","1,512",0,0,"1,512","1,512.27",유로,1512.27,1497.14,1527.39,20261007,2026-10-07T03:31:56.002Z
1,GBP,"1,765","1,800.65","1,782.83","1,782",0,0,"1,782","1,782.83",영국 파운드,1782.83,1765.0,1800.65,20261007,2026-10-07T03:31:56.002Z


## 🔧 Job Parameter 사용 가이드

### Job 등록 시 매개변수 설정

이 노트북을 Lakeflow Job에 등록할 때, **작업 > 매개변수** 탭에서 아래 key-value를 설정하면 widget 기본값을 덮어쓸 수 있습니다:

| Key | 설명 | 기본값 | 예시 |
| --- | --- | --- | --- |
| `search_date` | 조회 일자 (YYYYMMDD) | 빈값(=오늘) | `20261007` |
| `secret_scope` | Databricks Secret Scope | `exchange-rate-dev` | `exchange-rate-dev` |
| `secret_key` | Secret Key 이름 | `korea-exim-api-key` | `korea-exim-api-key` |
| `data_type` | 데이터 구분 | `AP01` | `AP01` (일별), `AP02` (추이) |
| `target_catalog` | 대상 카탈로그 | `workspace` | `workspace` |
| `target_schema` | 대상 스키마 | `default` | `default` |
| `target_table` | 대상 테이블명 | `daily_exchange_rate` | `daily_exchange_rate` |
| `write_mode` | 쓰기 모드 | `append` | `append`, `overwrite` |

### 🔒 API Key Secrets 설정 (최초 1회)

```bash
# 1. Secret Scope 생성 (Databricks CLI)
databricks secrets create-scope exchange-rate-dev

# 2. API Key를 Secret에 저장
databricks secrets put-secret exchange-rate-dev korea-exim-api-key --string-value "<API_KEY>"

# 3. Service Principal에 Secret Scope 권한 부여
databricks secrets put-acl exchange-rate-dev <SP_CLIENT_ID> READ
```

### 재사용 시나리오

* **일별 배치**: `search_date`를 비워두고 매일 실행 → 당일 환율 자동 수집
* **특정일 재수집**: `search_date=20261001` 주입 → 특정 일자 환율 재수집
* **환경 분리**: `target_catalog`를 dev/prod로 변경 → 동일 코드, 다른 저장소
* **전체 재적재**: `write_mode=overwrite` 주입 → 기존 데이터 교체

### API 출처

한국수출입은행 OpenAPI: `https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON`